# Plant Disease Detection - CNN Model Training

This notebook trains a **CNN-based model** for plant disease detection using the **PlantVillage** dataset.

## Overview
- **Dataset**: PlantVillage (38 classes: healthy + diseased leaves)
- **Architecture**: Transfer learning with **MobileNetV2** (lightweight, good for deployment)
- **Output**: Saved Keras model → `disease_module/model/plant_disease_cnn.keras`

## Prerequisites
```bash
pip install tensorflow tensorflow-datasets matplotlib
```

### 1. Import Libraries

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

print(f"TensorFlow: {tf.__version__}")

### 2. Load PlantVillage Dataset

In [ ]:
# Load PlantVillage from TensorFlow Datasets
(train_ds, val_ds), info = tfds.load(
    'plant_village',
    split=['train[:80%]', 'train[80%:]'],
    with_info=True,
    as_supervised=True
)

num_classes = info.features['label'].num_classes
class_names = info.features['label'].names
print(f"Classes: {num_classes}")
print(f"Sample: {class_names[:5]}...")

### 3. Preprocessing Pipeline

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    return image, label

train_ds = train_ds.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

### 4. Build CNN with Transfer Learning (MobileNetV2)

In [ ]:
# Base model: MobileNetV2 (pre-trained on ImageNet)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze base for transfer learning

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

### 5. Train the Model

In [ ]:
EPOCHS = 5  # Increase for better accuracy (e.g., 10-15)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

### 6. Save Model & Class Names

In [ ]:
MODEL_DIR = Path(__file__).parent / 'model' if '__file__' in dir() else Path('model')
MODEL_DIR = Path('model')  # Relative to notebook location
MODEL_DIR.mkdir(exist_ok=True)

model_path = MODEL_DIR / 'plant_disease_cnn.keras'
model.save(model_path)
print(f"Model saved to: {model_path.absolute()}")

# Save class names for predictor
import json
with open(MODEL_DIR / 'class_names.json', 'w') as f:
    json.dump(list(class_names), f, indent=2)
print(f"Class names saved ({len(class_names)} classes)")

### 7. Quick Test

In [ ]:
# Load one batch and predict
for images, labels in val_ds.take(1):
    preds = model.predict(images[:3])
    for i in range(3):
        idx = np.argmax(preds[i])
        print(f"True: {class_names[labels[i]]} | Pred: {class_names[idx]} ({preds[i][idx]:.2f})")